In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import numpy as np
sys.path.append(os.path.join(os.path.pardir, 'lesview'))
from lesview import *
from sbl_bbl import *

In [ ]:
H = 30
lat = 45.
f = coriolis(lat)
Ti = inertial_period(lat)

In [ ]:
casename = 'lsc_ymc22_sbl_bbl_v2'
figpath  = 'overview_{:s}'.format(casename)
os.makedirs(figpath, exist_ok=True)

In [ ]:
turbmethod = 'SMCLT-H15'
runs = {'r1': '', 'r2': '_rf'}
ds_ocgn = {}
for rkey in runs.keys():
    ocgn_dir = os.path.join(os.path.pardir, 'oceananigans', '{:s}{:s}'.format(casename, runs[rkey]))
    filepath = os.path.join(ocgn_dir, 'averages.jld2')
    ds_ocgn[rkey] = OceananigansDataProfile(filepath=filepath).dataset

In [ ]:
fig, axarr = plt.subplots(2, 1, sharex='col')
fig.set_size_inches(6,5)
levels=np.linspace(0,1.2,41)
abc = 'ab'
tags = ['Aligned', 'Opposite']
for i, rkey in enumerate(runs.keys()):
    ax = axarr[i]
    ds = ds_ocgn[rkey]
    tke = 0.5*(ds.data_vars['uu']+ds.data_vars['vv']+ds.data_vars['ww'].interp(zi=ds.z))
    tmp = f*H/np.sqrt(tke)*np.abs(ds.data_vars['v'].differentiate(coord='z')/ds.data_vars['u'].differentiate(coord='z'))
    # tmp = f*H/np.sqrt(tke)*np.abs(ds.data_vars['u'].differentiate(coord='z')/ds.data_vars['v'].differentiate(coord='z'))
    ratio = nondim_da(tmp, H, Ti)
    im = ratio.plot(ax=ax, levels=levels, cmap='pink_r', add_colorbar=False)
    if i == 1:
        ax.set_xlabel('$t/T_f$')
    else:
        ax.set_xlabel('')
    ax.set_ylabel('$z/H$')
    ax.set_xlim([0,16])
    ax.set_ylim([-1,0])
    ax.text(0.97, 0.06, tags[i], transform=ax.transAxes, va='bottom', ha='right')
    ax.text(0.03, 0.94, '({:s})'.format(abc[i]), transform=ax.transAxes, va='top', ha='left')
plt.subplots_adjust(top=0.97, bottom=0.12, left=0.12, right=0.82, hspace=0.15, wspace=0.15)
cax = plt.axes([0.86, 0.2, 0.015, 0.6])
cb = plt.colorbar(im, cax=cax)
cb.set_label('R')
figname = os.path.join(figpath, 'cross_gradient_flux_ratio')
fig.savefig(figname, dpi = 300, facecolor='w')